# Arbeitspaket (AP) 1: Management & Nutzung relationaler Daten

### Übungsprüfung «Hotelkette» (Musterlösung)

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[TT.MM.2026] / Raum: [anpassen] / [hh:mm – hh:mm]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung &amp; Design</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (hotel.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (hotel.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
    <tr><td>D</td><td><i>Zusatzübungen (freiwillig, nicht bewertet)</i></td><td>D1–D4</td><td>–</td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (z.B. Problem richtig benannt, aber falsch behoben; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `hotel.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — wird automatisch verwendet, falls keine PostgreSQL-Verbindung möglich ist.

**Tabellen:** `hotels`, `rooms`, `staff`, `guests`, `bookings`, `reviews`

**Datenbestand:** Buchungen mit Check-in in den Jahren 2024 und 2025. Datumswerte sind als Text im Format `YYYY-MM-DD` gespeichert.

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `hotel_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [1]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

Working directory: /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/Uebung/AP01_Uebung_Hotel


### Datenbankverbindung zu PostgreSQL

In [2]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
if host is not None:
    db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'
else:
    # Kein PostgreSQL erreichbar (z.B. lokal ohne Codespace) -> SQLite-Backup verwenden
    db_connection_url = 'sqlite:///hotel.db'
    print('Hinweis: Keine PostgreSQL-Verbindung gefunden - verwende SQLite-Backup hotel.db')

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Datenbank:', engine.dialect.name, '|', host if host else 'hotel.db')
engine.dispose()

Datenbank: postgresql | localhost


### Daten nach PostgreSQL laden

In [3]:
# Tabellen aus der SQLite-Datei hotel.db nach PostgreSQL übertragen
tables = ['hotels', 'rooms', 'staff', 'guests', 'bookings', 'reviews']

if db_connection_url.startswith('postgresql'):
    sqlite_engine = create_engine('sqlite:///hotel.db')
    engine = create_engine(db_connection_url)

    for table in tables:
        df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
        df.to_sql(table, engine, if_exists='replace', index=False)
        print(f'{table:<12}{len(df):>5} Zeilen')

    sqlite_engine.dispose()
    engine.dispose()
else:
    print('SQLite-Backup aktiv - die Daten liegen bereits in hotel.db.')

hotels          6 Zeilen
rooms          88 Zeilen
staff          36 Zeilen
guests        450 Zeilen
bookings     2380 Zeilen
reviews      1103 Zeilen


> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwendet das Setup automatisch die
> SQLite-Datenbank `hotel.db`. Sie können dies auch manuell erzwingen, indem Sie in den Code-Zellen
> ```python
> engine = create_engine(db_connection_url)
> ```
> ersetzen durch:
>
> ```python
> engine = create_engine('sqlite:///hotel.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Übung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `hotel.db` werden nicht gewertet.*

### A1 (4 Punkte)

Ein Entwickler schlägt vor, die Spalten `hotel_name` und `hotel_city` zusätzlich direkt in der Tabelle `bookings` zu speichern:

*«Dann brauchen wir für Auswertungen pro Hotel keine JOINs über `rooms` und `hotels` mehr.»*

Ist das eine gute Idee? Begründen Sie Ihre Antwort mit einem konkreten Szenario aus dem Hotelbetrieb (z.B. Umbenennung eines Hotels).

**Ihre Antwort:**

Nein, für die operative Datenbank ist das keine gute Idee. Name und Stadt hängen nicht von der Buchung ab, sondern vom Hotel. Sie sind über `bookings.roomid → rooms.hotelid → hotels` bereits eindeutig bestimmt. Würde man sie in `bookings` speichern, entstünde **Redundanz** (transitive Abhängigkeit, Verstoss gegen die 3. Normalform).

**Szenario (Update-Anomalie):** Die «City Lodge Oerlikon» wird nach einer Übernahme in «Hotel Oerlikon Park» umbenannt. Heute genügt die Änderung **einer** Zeile in `hotels`. Mit der vorgeschlagenen Lösung müssten zusätzlich alle **521 Buchungen** dieses Hotels angepasst werden. Wird dabei auch nur eine Zeile vergessen, existieren zwei Namen für dasselbe Hotel, und Auswertungen pro Hotel liefern inkonsistente Ergebnisse.

Die JOINs sind bei dieser Datenmenge kein Performance-Problem. Eine bewusste Denormalisierung kann höchstens in einer separaten Reporting- bzw. Data-Warehouse-Tabelle sinnvoll sein, nicht in der Quelldatenbank.

### A2 (4 Punkte)

In der Fremdschlüsselspalte `bookings.guestid` gibt es zwei Arten von auffälligen Werten:

- **(a)** 18 Buchungen mit `guestid` = `NULL` (Walk-in-Gäste ohne Gästeprofil)
- **(b)** 2 Buchungen mit `guestid` = `'G0999'` — diese ID existiert in der Tabelle `guests` nicht.

Welcher der beiden Fälle verletzt die **referentielle Integrität**? Begründen Sie Ihre Antwort und nennen Sie eine konkrete Auswirkung auf eine Auswertung (z.B. Umsatz pro Herkunftsland der Gäste).

**Ihre Antwort:**

Nur Fall **(b)** verletzt die referentielle Integrität. Diese verlangt, dass jeder Fremdschlüsselwert auf einen **existierenden** Primärschlüssel der referenzierten Tabelle verweist **oder NULL** ist (sofern die Beziehung optional ist, im ER-Diagramm `|o`).

- **(a)** `NULL` bedeutet «kein Gast zugeordnet». Das ist erlaubt und hier fachlich gewollt (Walk-in).
- **(b)** `'G0999'` verweist ins Leere: Es handelt sich um verwaiste Datensätze. Mit einem `FOREIGN KEY`-Constraint wäre dieser Insert von der Datenbank abgelehnt worden.

**Auswirkung:** Eine Auswertung «Umsatz pro Herkunftsland» verbindet `bookings` per `INNER JOIN` mit `guests`. Dabei fallen **beide** Fälle (20 Buchungen, CHF 3'885.00 Umsatz) stillschweigend weg, und der Gesamtumsatz wird zu tief ausgewiesen. Für (a) muss man bewusst entscheiden (z.B. `LEFT JOIN` und eigene Kategorie «Walk-in»). Für (b) ist der Datenfehler an der Quelle zu korrigieren.

### A3 (4 Punkte)

Bei zwei Buchungen in `bookings` liegt `checkout_date` **vor** `checkin_date` (z.B. `B00412`: Check-in 2024-04-23, Check-out 2024-04-22).

Wäre das **zwingend** ein Datenfehler? Begründen Sie in einem Satz und nennen Sie eine Massnahme, mit der die Datenbank solche Einträge künftig verhindern könnte.

**Ihre Antwort:**

Ja, zwingend: Ein Aufenthalt kann logisch nicht enden, bevor er begonnen hat. Anders als etwa ein Preis über dem Listenpreis gibt es dafür keinen denkbaren Geschäftsfall, die Daten sind also nicht **accurate** (vermutlich wurden die Felder bei der Erfassung vertauscht).

**Massnahme:** Ein `CHECK`-Constraint auf der Tabelle, z.B. `CHECK (checkout_date > checkin_date)`, und zusätzlich eine Validierung im Erfassungsformular.

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `hotel.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Anzahl Buchungen (alle Status) mit Check-in im Jahr **2025** pro Hotel — für **alle** Hotels in der Datenbank.

```sql
SELECT h.name, COUNT(*) AS bookings_2025
FROM hotels h
LEFT JOIN rooms r    ON h.hotelid = r.hotelid
LEFT JOIN bookings b ON r.roomid  = b.roomid
WHERE b.checkin_date BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY h.name;
```

Die Tabelle `hotels` enthält **6 Hotels**. Obwohl bereits `LEFT JOIN` verwendet wird, liefert die SQL-Abfrage nur **4 Zeilen** mit Buchungszahlen zwischen 200 und 300.

Das Management fragt: *«Zeig mir auch die Hotels, die 2025 keine Buchungen hatten.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 6 Hotels** erscheinen und ein Hotel ohne Buchungen 2025 korrekt mit `0` ausgewiesen wird. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

Zwei Änderungen sind nötig:

1. **Die Datumsbedingung gehört in die `ON`-Klausel, nicht in die `WHERE`-Klausel.** Für Hotels ohne Buchungen 2025 ist `b.checkin_date` nach dem `LEFT JOIN` entweder `NULL` (keine Buchung) oder ein Datum aus 2024. In beiden Fällen ist `BETWEEN '2025-01-01' AND '2025-12-31'` nicht `TRUE`, und `WHERE` entfernt die Zeile. Eine `WHERE`-Bedingung auf die rechte Tabelle macht den `LEFT JOIN` damit faktisch zu einem `INNER JOIN`. Deshalb fehlen «Hotel Alpenblick» (nur Buchungen 2024, Renovation 2025) und «Hotel Bellevue Lago» (Eröffnung 2026). In der `ON`-Klausel bestimmt die Bedingung nur, **welche Buchungen angehängt werden**, nicht welche Hotels im Ergebnis bleiben.

2. **`COUNT(*)` muss zu `COUNT(b.bookingid)` werden.** `COUNT(*)` zählt Zeilen. Wegen des `LEFT JOIN` über `rooms` hat ein Hotel ohne Buchungen **eine Zeile pro Zimmer** (mit `NULL` in allen `bookings`-Spalten). Mit `COUNT(*)` würden «Hotel Alpenblick» und «Hotel Bellevue Lago» darum fälschlich mit **12** (= Anzahl Zimmer) statt mit 0 erscheinen. `COUNT(b.bookingid)` ignoriert `NULL` und liefert korrekt `0`.

*Hinweis zur Bewertung: Wer nur auf eine der beiden Änderungen kommt, erhält ein falsches Ergebnis (4 Zeilen bzw. 12 statt 0). Ein `ORDER BY` ist nicht verlangt.*

*Alternativer Lösungsweg: die Buchungen 2025 zuerst in einer Subquery pro `hotelid` zählen und diese per `LEFT JOIN` an `hotels` hängen, mit `COALESCE(anzahl, 0)`.*

**Korrigierte SQL-Abfrage:**

In [4]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
SELECT
    h.name,
    COUNT(b.bookingid) AS bookings_2025
FROM hotels h
LEFT JOIN rooms r    ON h.hotelid = r.hotelid
LEFT JOIN bookings b ON r.roomid  = b.roomid
                    AND b.checkin_date BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY h.name
ORDER BY bookings_2025 DESC
""", con=engine)

engine.dispose()
df_B1

,name,bookings_2025
0,Hotel Limmatblick,300
1,Hotel Rheinufer,257
2,City Lodge Oerlikon,248
3,Hotel Aarehof,207
4,Hotel Alpenblick,0
5,Hotel Bellevue Lago,0


### B2 (8 Punkte)

**Ziel:** Übersicht pro Hotel mit der Anzahl Mitarbeitenden (`staff_count`) und dem Umsatz aus abgeschlossenen Buchungen (`revenue`, nur `status = 'Completed'`) — für alle Hotels mit mindestens einer abgeschlossenen Buchung.

```sql
SELECT h.name,
       COUNT(s.staffid)  AS staff_count,
       SUM(b.total_paid) AS revenue
FROM hotels h
JOIN staff s    ON h.hotelid = s.hotelid
JOIN rooms r    ON h.hotelid = r.hotelid
JOIN bookings b ON r.roomid  = b.roomid
WHERE b.status = 'Completed'
GROUP BY h.name;
```

Tatsächliches Ergebnis (Auszug):

| name | staff_count | revenue |
|---|---|---|
| Hotel Limmatblick | 4184 | 1925339.28 |
| Hotel Rheinufer | 3101 | 1470813.33 |
| City Lodge Oerlikon | 2195 | 723090.85 |

Die Tabelle `staff` enthält insgesamt nur **36 Mitarbeitende**, und das Controlling meldet für das Hotel Limmatblick einen Umsatz von rund **CHF 240'000**.

**Aufgabe:** Erklären Sie, warum **beide** Kennzahlen falsch sind, und schreiben Sie die korrigierte SQL-Abfrage. Runden Sie `revenue` mit `ROUND(CAST(... AS NUMERIC), 2)` auf 2 Dezimalstellen und sortieren Sie absteigend nach `revenue`. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

**Ursache: Join-Explosion (Fan-out).** `staff` und `bookings` (über `rooms`) stehen beide in einer **1:n-Beziehung** zu `hotels`, aber in keiner Beziehung zueinander. Werden beide in derselben Abfrage an `hotels` gejoint, kombiniert die Datenbank pro Hotel **jede** Buchung mit **jedem** Mitarbeitenden (kartesisches Produkt je Hotel):

- Hotel Limmatblick: 523 abgeschlossene Buchungen × 8 Mitarbeitende = **4184 Zeilen**. `COUNT(s.staffid)` zählt deshalb 4184 statt 8.
- Jeder Buchungsbetrag erscheint 8-mal, also `SUM(b.total_paid)` = 8 × 240'667.41 = **1'925'339.28**.

**Korrektur:** Jede 1:n-Beziehung wird **separat in einer Subquery voraggregiert** (eine Zeile pro `hotelid`), und erst danach werden die Ergebnisse über `hotelid` an `hotels` gejoint. Diese Joins sind 1:1, also kann nichts mehr vervielfacht werden. Der Statusfilter wandert in die Umsatz-Subquery, weil er nur die Buchungen betrifft. Zusätzlich wird gemäss Aufgabenstellung gerundet und sortiert.

*Warum `COUNT(DISTINCT s.staffid)` allein nicht reicht: Es korrigiert nur `staff_count`, der Umsatz bliebe 8-fach. `SUM(DISTINCT b.total_paid)` wäre ebenfalls falsch, weil gleiche Beträge verschiedener Buchungen dann nur einmal gezählt würden. Solche Lösungen gelten als «Problem erkannt, aber falsch behoben».*

*Alternative: dieselbe Logik mit `WITH`-Klauseln (CTEs) statt Subqueries im `FROM`.*

**Korrigierte SQL-Abfrage:**

In [5]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
SELECT
    h.name,
    st.staff_count,
    rv.revenue
FROM hotels h
JOIN (SELECT hotelid, COUNT(*) AS staff_count
      FROM staff
      GROUP BY hotelid) AS st
     ON h.hotelid = st.hotelid
JOIN (SELECT r.hotelid, ROUND(CAST(SUM(b.total_paid) AS NUMERIC), 2) AS revenue
      FROM rooms r
      JOIN bookings b ON r.roomid = b.roomid
      WHERE b.status = 'Completed'
      GROUP BY r.hotelid) AS rv
     ON h.hotelid = rv.hotelid
ORDER BY rv.revenue DESC
""", con=engine)

engine.dispose()
df_B2

,name,staff_count,revenue
0,Hotel Limmatblick,8,240667.41
1,Hotel Rheinufer,7,210116.19
2,Hotel Alpenblick,4,159054.65
3,City Lodge Oerlikon,5,144618.17
4,Hotel Aarehof,6,102521.28


---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `hotel.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Die gültigen Werte in `guests.country` sind `Schweiz`, `Deutschland`, `Österreich`, `Italien` und `Frankreich`. Schreiben Sie eine SQL-Abfrage, die alle Gäste (`guestid`, `firstname`, `lastname`, `country`) mit einem **ungültigen oder fehlenden** Wert in `country` ausgibt, sortiert nach `guestid`. Begründen Sie Ihre Filterbedingung.

**Begründung der Vorgehensweise:**

Alle Spalten liegen in `guests`, ein JOIN ist nicht nötig. `NOT IN (...)` findet alle Werte, die keinem der fünf gültigen Werte **exakt** entsprechen. Damit werden Kleinschreibung (`schweiz`, `italien`), Länderkürzel (`CH`, `DE`), englische Bezeichnungen (`Switzerland`) und sogar `'Deutschland '` mit einem **Leerzeichen am Ende** erfasst.

**Wichtig:** `NOT IN` erfasst **keine `NULL`-Werte**. Der Vergleich `NULL NOT IN (...)` ergibt weder `TRUE` noch `FALSE`, sondern `UNKNOWN`, und `WHERE` behält nur `TRUE`-Zeilen. Fehlende Werte müssen deshalb mit `OR country IS NULL` explizit ergänzt werden. Ohne diese Bedingung liefert die Abfrage nur 6 statt 9 Gäste.

*Tipp: Versteckte Leerzeichen erkennt man z.B. mit `SELECT country, LENGTH(country) FROM guests`.*

*Hinweis zur Bewertung: fehlendes `OR country IS NULL` gilt als falsche Filterung (mittlerer Mangel).*

**SQL-Abfrage:**

In [6]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
SELECT
    guestid,
    firstname,
    lastname,
    country
FROM guests
WHERE country NOT IN ('Schweiz', 'Deutschland', 'Österreich', 'Italien', 'Frankreich')
   OR country IS NULL
ORDER BY guestid
""", con=engine)

engine.dispose()
df_C1

,guestid,firstname,lastname,country
0,G0037,Fabian,Vogel,schweiz
1,G0081,Jonas,Bühler,CH
2,G0126,Elias,Gerber,Switzerland
3,G0170,Nina,Brunner,Deutschland
4,G0214,Lara,Lehmann,DE
5,G0259,Anna,Brunner,italien
6,G0302,Chiara,Meyer,NaN
7,G0345,Julia,Brunner,NaN
8,G0391,Sophie,Frei,NaN


### C2 (4 Punkte)

Wie hoch ist der durchschnittlich bezahlte **Preis pro Nacht** je Zimmertyp (`room_type`)? Berücksichtigen Sie nur abgeschlossene Buchungen (`status = 'Completed'`). Berechnen Sie den Preis pro Nacht als Gesamtumsatz (`SUM(total_paid)`) geteilt durch die Gesamtzahl der Nächte (`SUM(nights)`).

Geben Sie `room_type`, die Anzahl Buchungen (`booking_count`) und den Preis pro Nacht (`avg_price_per_night`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`. Sortieren Sie absteigend nach `avg_price_per_night`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum der Statusfilter nötig ist.

**Begründung der Vorgehensweise:**

`room_type` liegt in `rooms`, `total_paid` und `nights` liegen in `bookings`. Ein JOIN über `roomid` verbindet beide Tabellen, und `GROUP BY r.room_type` fasst die Buchungen je Zimmertyp zusammen. Ein `INNER JOIN` ist korrekt, weil Zimmer ohne Buchung keinen Preis haben.

**Statusfilter:** Stornierte Buchungen haben `total_paid = 0`, aber `nights > 0`, und bei No-Shows wird nur eine Nacht verrechnet. Ohne `WHERE b.status = 'Completed'` würde der Preis pro Nacht deutlich zu tief ausgewiesen (z.B. Suite 259.85 statt 295.88).

*Hinweis: `AVG(b.total_paid / b.nights)` ist ein anderes, ungewichtetes Mittel (Suite: 297.32) und entspricht nicht der vorgegebenen Berechnung.*

**SQL-Abfrage:**

In [7]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
SELECT
    r.room_type,
    COUNT(b.bookingid) AS booking_count,
    ROUND(CAST(SUM(b.total_paid) / SUM(b.nights) AS NUMERIC), 2) AS avg_price_per_night
FROM rooms r
JOIN bookings b ON r.roomid = b.roomid
WHERE b.status = 'Completed'
GROUP BY r.room_type
ORDER BY avg_price_per_night DESC
""", con=engine)

engine.dispose()
df_C2

,room_type,booking_count,avg_price_per_night
0,Suite,168,295.88
1,Family,350,190.41
2,Double,917,154.86
3,Single,601,122.90


### C3 (4 Punkte)

Welche Hotels haben eine durchschnittliche Gästebewertung (`rating`) **unter 4.0**? Geben Sie `name`, `city`, die Anzahl Bewertungen (`review_count`) und die Durchschnittsbewertung (`avg_rating`, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`) aus, **aufsteigend** sortiert nach `avg_rating`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

`reviews` ist nicht direkt mit `hotels` verknüpft. Eine Bewertung gehört zu einer Buchung, die Buchung zu einem Zimmer und das Zimmer zu einem Hotel. Es braucht deshalb drei JOINs: `hotels → rooms → bookings → reviews`. `GROUP BY h.name, h.city` fasst alle Bewertungen pro Hotel zusammen.

Weil auf ein **Aggregat** gefiltert wird, muss die Bedingung in `HAVING` stehen, nicht in `WHERE` (`WHERE` wird vor der Gruppierung ausgewertet). In `HAVING` wird der Ausdruck `AVG(rv.rating)` wiederholt, weil PostgreSQL dort keine Spalten-Aliase erlaubt.

*Hinweis: «Hotel Rheinufer» liegt mit 4.03 knapp über der Grenze und gehört nicht ins Ergebnis.*

**SQL-Abfrage:**

In [8]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
SELECT
    h.name,
    h.city,
    COUNT(rv.reviewid) AS review_count,
    ROUND(CAST(AVG(rv.rating) AS NUMERIC), 2) AS avg_rating
FROM hotels h
JOIN rooms r     ON h.hotelid   = r.hotelid
JOIN bookings b  ON r.roomid    = b.roomid
JOIN reviews rv  ON b.bookingid = rv.bookingid
GROUP BY h.name, h.city
HAVING AVG(rv.rating) < 4.0
ORDER BY avg_rating ASC
""", con=engine)

engine.dispose()
df_C3

,name,city,review_count,avg_rating
0,City Lodge Oerlikon,Zürich,223,3.72
1,Hotel Aarehof,Bern,204,3.92


---
## Teil D – Zusatzübungen (freiwillig, nicht Teil der 40 Punkte)

*Weitere typische Fallen zum Üben. D1–D2: Fehleranalyse wie Teil B. D3–D4: Abfragen schreiben wie Teil C.*

### D1 – Fehleranalyse

**Ziel:** Anzahl stornierter oder nicht wahrgenommener Buchungen (`status` = `'Cancelled'` **oder** `'No-Show'`) pro Hotel — **nur für Hotels in Zürich**.

```sql
SELECT h.name, h.city, COUNT(*) AS booking_count
FROM hotels h
JOIN rooms r    ON h.hotelid = r.hotelid
JOIN bookings b ON r.roomid  = b.roomid
WHERE b.status = 'Cancelled' OR b.status = 'No-Show' AND h.city = 'Zürich'
GROUP BY h.name, h.city;
```

Tatsächliches Ergebnis: **5 Zeilen**, darunter auch Hotels in Bern, Basel und Davos.

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage (absteigend sortiert nach `booking_count`) und begründen Sie die Änderung.

**Begründung der Änderungen:**

`AND` bindet stärker als `OR`. Die Bedingung wird deshalb gelesen als

`b.status = 'Cancelled'` **OR** `(b.status = 'No-Show' AND h.city = 'Zürich')`

Stornierte Buchungen werden also aus **allen** Städten gezählt, und nur bei den No-Shows greift der Zürich-Filter. Die Korrektur: die beiden Status-Bedingungen klammern, z.B. `(b.status = 'Cancelled' OR b.status = 'No-Show') AND h.city = 'Zürich'`, oder wie unten mit `IN` schreiben. Das ist kürzer und vermeidet das Problem ganz.

**Korrigierte SQL-Abfrage:**

In [9]:
engine = create_engine(db_connection_url)

df_D1 = pd.read_sql_query("""
SELECT
    h.name,
    h.city,
    COUNT(b.bookingid) AS booking_count
FROM hotels h
JOIN rooms r    ON h.hotelid = r.hotelid
JOIN bookings b ON r.roomid  = b.roomid
WHERE b.status IN ('Cancelled', 'No-Show')
  AND h.city = 'Zürich'
GROUP BY h.name, h.city
ORDER BY booking_count DESC
""", con=engine)

engine.dispose()
df_D1

,name,city,booking_count
0,Hotel Limmatblick,Zürich,92
1,City Lodge Oerlikon,Zürich,82


### D2 – Fehleranalyse

**Ziel:** Liste aller Gäste (`guestid`, `firstname`, `lastname`), die noch **nie** gebucht haben.

```sql
SELECT g.guestid, g.firstname, g.lastname
FROM guests g
WHERE g.guestid NOT IN (SELECT b.guestid FROM bookings b);
```

Tatsächliches Ergebnis: **0 Zeilen** — obwohl es Gäste ohne Buchung gibt (z.B. die zuletzt registrierten Gäste).

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage (sortiert nach `guestid`) und begründen Sie die Änderung. *(Tipp: Schauen Sie sich die Spalte `bookings.guestid` genauer an.)*

**Begründung der Änderungen:**

Die Subquery `SELECT b.guestid FROM bookings b` liefert auch die **`NULL`-Werte** der 18 Walk-in-Buchungen. `x NOT IN (..., NULL)` bedeutet `x <> ... AND ... AND x <> NULL`. Der Vergleich mit `NULL` ergibt immer `UNKNOWN`, also ist die Gesamtbedingung **nie `TRUE`**, und es wird keine einzige Zeile zurückgegeben.

Korrekturmöglichkeiten (alle gleichwertig):

1. `NOT EXISTS` (unten): prüft pro Gast, ob eine passende Buchung existiert, und ist robust gegenüber `NULL`.
2. `NOT IN (SELECT b.guestid FROM bookings b WHERE b.guestid IS NOT NULL)`
3. `LEFT JOIN bookings b ON g.guestid = b.guestid WHERE b.bookingid IS NULL`

Ergebnis: 43 Gäste ohne Buchung.

**Korrigierte SQL-Abfrage:**

In [10]:
engine = create_engine(db_connection_url)

df_D2 = pd.read_sql_query("""
SELECT
    g.guestid,
    g.firstname,
    g.lastname
FROM guests g
WHERE NOT EXISTS (SELECT 1
                  FROM bookings b
                  WHERE b.guestid = g.guestid)
ORDER BY g.guestid
""", con=engine)

engine.dispose()
df_D2

,guestid,firstname,lastname
0,G0042,Sara,Frei
1,G0069,Robin,Widmer
2,G0080,Julia,Meyer
3,G0106,Julia,Gerber
4,G0116,Lara,Brunner
5,G0118,Jonas,Weber
6,G0134,Lara,Bachmann
7,G0167,Simon,Wyss
8,G0169,Nico,Gerber
9,G0183,Matteo,Fischer


### D3 – Referentielle Integrität prüfen

Schreiben Sie eine SQL-Abfrage, die alle Buchungen (`bookingid`, `guestid`, `checkin_date`, `total_paid`) ausgibt, deren `guestid` **nicht leer** ist, aber auf **keinen existierenden Gast** verweist (vgl. A2). Begründen Sie Ihre Vorgehensweise.

**Begründung der Vorgehensweise:**

Ein `LEFT JOIN` von `bookings` auf `guests` behält alle Buchungen. Wenn keine passende Zeile in `guests` existiert, ist `g.guestid` danach `NULL`. Zusätzlich muss `b.guestid IS NOT NULL` gelten, sonst würden auch die 18 erlaubten Walk-in-Buchungen (`guestid` = `NULL`) ausgegeben.

*Alternative: `WHERE b.guestid IS NOT NULL AND NOT EXISTS (SELECT 1 FROM guests g WHERE g.guestid = b.guestid)`.*

**SQL-Abfrage:**

In [11]:
engine = create_engine(db_connection_url)

df_D3 = pd.read_sql_query("""
SELECT
    b.bookingid,
    b.guestid,
    b.checkin_date,
    b.total_paid
FROM bookings b
LEFT JOIN guests g ON b.guestid = g.guestid
WHERE b.guestid IS NOT NULL
  AND g.guestid IS NULL
ORDER BY b.bookingid
""", con=engine)

engine.dispose()
df_D3

,bookingid,guestid,checkin_date,total_paid
0,B01564,G0999,2025-03-14,270.0
1,B02056,G0999,2025-09-02,270.0


### D4 – Plausibilitätsprüfung

Schreiben Sie eine SQL-Abfrage, die alle Buchungen ausgibt, bei denen `checkout_date` **nicht nach** `checkin_date` liegt (vgl. A3). Geben Sie `bookingid`, den Hotelnamen (`name`), `checkin_date`, `checkout_date` und `nights` aus. Begründen Sie, warum der Vergleich der Datumsspalten hier funktioniert, obwohl sie als Text gespeichert sind.

**Begründung der Vorgehensweise:**

Der Hotelname liegt in `hotels`, also braucht es den Weg `bookings → rooms → hotels`. Der Filter `checkout_date <= checkin_date` findet alle unplausiblen Aufenthalte (Abreise vor oder am Anreisetag).

Der Textvergleich funktioniert, weil die Daten im **ISO-Format `YYYY-MM-DD`** gespeichert sind. Bei gleicher Länge und absteigender Wertigkeit (Jahr → Monat → Tag) entspricht die alphabetische Reihenfolge genau der chronologischen. Bei einem Format wie `DD.MM.YYYY` wäre das **nicht** der Fall.

**SQL-Abfrage:**

In [12]:
engine = create_engine(db_connection_url)

df_D4 = pd.read_sql_query("""
SELECT
    b.bookingid,
    h.name,
    b.checkin_date,
    b.checkout_date,
    b.nights
FROM bookings b
JOIN rooms r  ON b.roomid  = r.roomid
JOIN hotels h ON r.hotelid = h.hotelid
WHERE b.checkout_date <= b.checkin_date
ORDER BY b.bookingid
""", con=engine)

engine.dispose()
df_D4

,bookingid,name,checkin_date,checkout_date,nights
0,B00412,Hotel Limmatblick,2024-04-23,2024-04-22,1
1,B01703,Hotel Limmatblick,2025-05-09,2025-05-07,2


---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [13]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-31 15:04:07
Python Version: 3.12.11
IP Address: 127.0.0.1
-----------------------------------
